In [ ]:
import pandas as pd
from pathlib import Path
from scipy.stats import ttest_ind
import plotly.graph_objects as go

In [ ]:
BIOMARKER = "Vitamin D"
INTERVENTION_DATE = "2026-01-01"

In [ ]:
rythm_dir = Path("data/rythm")
df = pd.concat(
    [pd.read_csv(csv_path) for csv_path in sorted(rythm_dir.glob("*.csv"))],
    ignore_index=True,
)
df["time"] = pd.to_datetime(df["time"])
df["value"] = pd.to_numeric(df["value"], errors="coerce")

In [ ]:
biomarker_df = df[df["marker"] == BIOMARKER].sort_values("time").copy()
if biomarker_df.empty:
    raise ValueError(f"No rows found for biomarker '{BIOMARKER}'")

intervention = pd.to_datetime(INTERVENTION_DATE)
before_values = biomarker_df.loc[biomarker_df["time"] < intervention, "value"]
after_values = biomarker_df.loc[biomarker_df["time"] >= intervention, "value"]

if before_values.empty or after_values.empty:
    raise ValueError(
        f"Need at least one measurement before and after {INTERVENTION_DATE}. "
        f"Found {len(before_values)} before and {len(after_values)} after."
    )

mean_before = before_values.mean()
mean_after = after_values.mean()
delta = mean_after - mean_before
t_stat, p_value = ttest_ind(before_values, after_values, equal_var=False)

print(f"Pre-intervention mean: {mean_before:.4g} (n={len(before_values)})")
print(f"Post-intervention mean: {mean_after:.4g} (n={len(after_values)})")
print(f"Delta: {delta:.4g}")
print(f"t = {t_stat:.4g}, p = {p_value:.4g}")

In [ ]:
label_font = dict(size=14, family="Arial Black, Arial, sans-serif", color="#111827")

unit = biomarker_df["unit"].iloc[0]
x_min = biomarker_df["time"].min()
x_max = biomarker_df["time"].max()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=biomarker_df["time"],
        y=biomarker_df["value"],
        mode="markers+lines",
        name=BIOMARKER,
        line=dict(color="#2563eb", width=2),
        marker=dict(size=7, color="#2563eb", line=dict(width=1, color="white")),
    )
)

fig.add_shape(
    type="line",
    x0=intervention,
    x1=intervention,
    y0=0,
    y1=1,
    yref="paper",
    line=dict(color="#dc2626", dash="dash", width=1.5),
)
fig.add_annotation(
    x=intervention,
    y=1,
    yref="paper",
    text="Intervention start",
    showarrow=False,
    yanchor="bottom",
    font=label_font,
)

fig.add_shape(
    type="line",
    x0=x_min,
    x1=intervention,
    y0=mean_before,
    y1=mean_before,
    line=dict(color="#64748b", dash="dot", width=1.5),
)
fig.add_shape(
    type="line",
    x0=intervention,
    x1=x_max,
    y0=mean_after,
    y1=mean_after,
    line=dict(color="#059669", dash="dot", width=1.5),
)

fig.add_annotation(
    x=intervention,
    y=mean_before,
    text=f"Pre: {mean_before:.2g}",
    showarrow=False,
    xanchor="right",
    yanchor="bottom",
    font=dict(size=14, family="Arial Black, Arial, sans-serif", color="#64748b"),
)
fig.add_annotation(
    x=x_max,
    y=mean_after,
    text=f"Post: {mean_after:.2g} (Δ={delta:.2g}, p={p_value:.3g})",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font=dict(size=14, family="Arial Black, Arial, sans-serif", color="#059669"),
)

fig.update_layout(
    title=dict(text=f"{BIOMARKER} over time", font=dict(size=14)),
    xaxis_title=dict(text="Date", font=label_font),
    yaxis_title=f"{BIOMARKER} ({unit})",
    width=960,
    height=480,
    margin=dict(l=60, r=30, t=50, b=45),
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(color="#111827", size=11),
    legend=dict(bgcolor="white", borderwidth=0),
    xaxis=dict(
        showgrid=True,
        gridcolor="#f3f4f6",
        linecolor="#e5e7eb",
        tickfont=label_font,
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="#f3f4f6",
        linecolor="#e5e7eb",
        tickfont=dict(size=10),
    ),
)
fig.show()